# Spatiotemporal dynamics

This simulation is included in the scBIOT Figshare collection and contains counts, spatial coordinates, ordered time points, and lineage labels. The notebook uses the v1.2.0 linear autoencoder before spatial/time-aware OT.

- Collection: [AnnData for scBIOT analysis](https://figshare.com/articles/dataset/Anndata_for_scBIOT_analysis/30671669)
- Direct file: [sim_spatial.h5ad](https://ndownloader.figshare.com/files/66868580)


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"


In [ ]:
path = fetch("sim_spatial.h5ad", "https://ndownloader.figshare.com/files/66868580")
adata = subsample(sc.read_h5ad(path), 12_000)
adata.obs["batch"] = adata.obs["timepoint"].astype(str)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
adata


## Spatial/time-aware integration


In [ ]:
adata = scb.pp.autoencoder(
    adata, input_key="counts", out_key="X_ae", batch_key="batch",
    n_top_genes=min(300, adata.n_vars), latent_dim=20, max_epochs=AE_EPOCHS,
    early_stop_patience=5, random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata, obsm_key="X_ae", batch_key="batch", out_key="X_scbiot_st",
    spatial_key="spatial", spatial_weight=0.5,
    time_key="timepoint", time_weight=0.5, time_mode="auto",
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


## Velocity, transport genes, and energy


In [ ]:
adata = scb.tl.velocity_field_sb_centroids(
    adata, obsm_key="X_scbiot_st", spatial_key="spatial",
    time_key="timepoint", lineage_key="lineage", out_vel_key="velocity_sb",
    time_bins=None, n_centroids_per_bin=64, max_samples_per_bin=20_000,
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
ranked = scb.tl.rank_transport_score(
    adata, time_key="timepoint", rep_key="X_scbiot_st",
    store_key="transport_score", n_perms=20, random_state=RANDOM_STATE,
)
time_levels = list(adata.obs["timepoint"].cat.categories)
first_step = scb.tl.rank_transport_score(
    adata, cond1=time_levels[0], cond2=time_levels[1], cond_key="timepoint",
    rep_key="X_scbiot_st", store_key="first_step_transport",
    n_perms=20, random_state=RANDOM_STATE,
)
scb.tl.compute_transport_energy(
    adata, layer="transport_fwd", key_added="transport_energy", log1p=True
)
ranked.head(10)


In [ ]:
sc.pp.neighbors(adata, use_rep="X_scbiot_st", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["timepoint", "lineage", "transport_energy"])
